# LC 373 — Find K Pairs with Smallest Sums
**Day-72 | Heap / Priority Queue | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Use a min-heap seeded with the first
<em>k</em> pairs <code>(nums1[0], nums2[j])</code>. Each pop yields
the next smallest pair; advance the <em>nums1</em> pointer to explore
the next candidate — avoiding a brute-force O(n²) scan.
</div>

## Official Problem Statement

Given two integer arrays `nums1` and `nums2` sorted in **non-decreasing
order** and an integer `k`, return the **k pairs** `(u, v)` with the
**smallest sums** where `u` is from `nums1` and `v` is from `nums2`.

Return the pairs in any order.

**Constraints:**
- `1 <= nums1.length, nums2.length <= 10^5`
- `-10^9 <= nums1[i], nums2[i] <= 10^9`
- `1 <= k <= 10^4`

**Examples:**
```
nums1=[1,7,11], nums2=[2,4,6], k=3
Output: [[1,2],[1,4],[1,6]]

nums1=[1,1,2], nums2=[1,2,3], k=2
Output: [[1,1],[1,1]]
```

## What This Is Actually Asking

Imagine a grid where rows = nums1, cols = nums2.
Each cell `(i, j)` has value `nums1[i] + nums2[j]`.

```
      2   4   6      ← nums2
 1  [ 3   5   7 ]
 7  [ 9  11  13 ]
11  [13  15  17 ]
↑ nums1
```

We want the k cells with the smallest values. But n × m can be 10^10 —
we cannot enumerate all cells. The heap lets us explore only the
promising frontier.

## Walk Through an Example by Hand

`nums1=[1,7,11]`, `nums2=[2,4,6]`, `k=3`

**Seed heap** with `(nums1[0]+nums2[j], 0, j)` for j in 0..k-1:
```
heap = [(3,0,0), (5,0,1), (7,0,2)]
```

| Pop           | Result pair | Push next (i+1, j) |
|---------------|-------------|--------------------|
| (3, 0, 0)     | [1, 2]      | (9, 1, 0)          |
| (5, 0, 1)     | [1, 4]      | (11, 1, 1)         |
| (7, 0, 2)     | [1, 6]      | (13, 1, 2)         |

After 3 pops we have k=3 results: `[[1,2],[1,4],[1,6]]` ✓

Each pop produces one result; each push explores one new candidate.

## The Picture

```
Grid (row=nums1 index, col=nums2 index), sums shown:

      j=0  j=1  j=2
i=0 [  3    5    7  ]  ← all seeded into heap initially
i=1 [  9   11   13  ]  ← pushed when (i=0, j) is popped
i=2 [ 13   15   17  ]  ← pushed when (i=1, j) is popped

Heap state evolution:

Initial seed (j=0..k-1 with i=0):
  heap: [(3,0,0), (5,0,1), (7,0,2)]
          ^
          smallest sum first

Pop (3,0,0) → result [1,2]; push (9,1,0):
  heap: [(5,0,1), (7,0,2), (9,1,0)]

Pop (5,0,1) → result [1,4]; push (11,1,1):
  heap: [(7,0,2), (9,1,0), (11,1,1)]

Key invariant: heap always holds the cheapest unexplored
  candidates — one per column j that we've touched.
```

## When To Use This Pattern

Use a **seeded min-heap with frontier expansion** when:

- You have a 2D (or multi-D) sorted space too large to enumerate
- You need the **k smallest** combinations from sorted sources
- Each cell's neighbors are predictably larger (sorted arrays)

**Signals in the problem:**
- "k pairs / tuples with smallest / largest sums"
- "sorted arrays, find k combinations"
- "merge k sorted lists"

**Related problems:** LC 378 (k-th smallest in sorted matrix),
LC 23 (merge k sorted lists)

## The Approach

**Min-Heap with Frontier Expansion**

```
1. Seed heap: for j in 0..min(k,len(nums2))-1:
     push (nums1[0]+nums2[j], 0, j)

2. Repeat up to k times:
   a. Pop (sum, i, j) — smallest current candidate
   b. Append [nums1[i], nums2[j]] to results
   c. If i+1 < len(nums1):
        push (nums1[i+1]+nums2[j], i+1, j)

3. Return results
```

Why seed with row 0 only: since nums1 is sorted, for any fixed
column j the cheapest row-0 pair must be explored before row 1+.

**Time:** O(k log k) | **Space:** O(k)

In [ ]:
import heapq
from typing import List

In [ ]:
def test_harness(func):
    cases = [
        # (nums1, nums2, k, expected_pairs_any_order)
        (
            [1, 7, 11], [2, 4, 6], 3,
            [[1, 2], [1, 4], [1, 6]]
        ),
        (
            [1, 1, 2], [1, 2, 3], 2,
            [[1, 1], [1, 1]]
        ),
        (
            [1, 2], [3], 3,
            [[1, 3], [2, 3]]
        ),
        (
            [1, 2, 4, 5, 6], [3, 5, 7, 9], 3,
            [[1, 3], [2, 3], [1, 5]]
        ),
    ]
    passed = 0
    for nums1, nums2, k, expected in cases:
        result = func(nums1[:], nums2[:], k)
        # sort both for order-independent comparison
        r_sorted = sorted(sorted(p) for p in result)
        e_sorted = sorted(sorted(p) for p in expected)
        status = "PASSED" if r_sorted == e_sorted else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status} | n1={nums1}, n2={nums2}, k={k}"
                f"\n    expected={expected}"
                f"\n    got     ={result}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")

In [ ]:
def k_smallest_pairs(
    nums1: List[int],
    nums2: List[int],
    k: int
) -> List[List[int]]:
    """
    Return k pairs (u, v) with smallest sums, u from nums1,
    v from nums2 (both sorted ascending).

    Strategy: seed heap with (nums1[0]+nums2[j], 0, j) for
      j in 0..min(k, len(nums2))-1. On each pop of (s, i, j)
      record [nums1[i], nums2[j]]; if i+1 < len(nums1) push
      the next row candidate for column j.

    Args:
        nums1: Sorted list of integers.
        nums2: Sorted list of integers.
        k:     Number of pairs to return.

    Returns:
        List of k [u, v] pairs with smallest sums.

    Time:  O(k log k)
    Space: O(k)
    """
    results: List[List[int]] = []
    heap: List = []

    # Seed: first row paired with first k columns
    # --- debug: print(f"Seeding heap...")
    pass  # TODO: seed heap

    while heap and len(results) < k:
        # --- debug: print(f"heap top={heap[0]}, results={results}")
        pass  # TODO: pop, record, push next

    return results

In [ ]:
# Uncomment and run when solution is ready
# test_harness(k_smallest_pairs)

## Complexity

| Dimension | Value    | Reason                                       |
|-----------|----------|----------------------------------------------|
| Time      | O(k log k) | k pops/pushes; each heap op costs O(log k) |
| Space     | O(k)     | heap holds at most k entries                 |

**Comparison:**

| Approach             | Time         | Space  | Notes                    |
|----------------------|--------------|--------|---------------------------|
| Brute force all pairs | O(nm log nm) | O(nm)  | infeasible for large n,m |
| Heap frontier         | O(k log k)   | O(k)   | optimal; uses sorted order|

The brute-force generates up to 10^10 pairs; the heap explores only
O(k) candidates, leveraging the sorted structure of both arrays.

## Real World Connection

**Flight / travel booking search engines** use this exact pattern.

- Kayak or Google Flights must find the k cheapest itineraries
  combining outbound legs (nums1) and return legs (nums2) from
  sorted price lists. Enumerating all combinations is infeasible.
  A seeded min-heap produces the k cheapest in O(k log k).

- **E-commerce bundle pricing**: find the k cheapest product
  bundles by combining items from two sorted price catalogs.

- **Database query optimization**: merge-join two sorted result
  sets and return only the top-k rows by a computed score —
  the heap avoids materializing the full cross product.

Whenever you have two sorted sources and need the best k
combinations, frontier expansion with a min-heap is the
production-grade solution.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra